# What public data allow us to claim about violence against LGBT+ people in Brazil: replication

**Author:** Carlos Ulisses Flores ([ORCID 0000-0002-6034-7765](https://orcid.org/0000-0002-6034-7765)),
Codex Hash Research Laboratory ·
**Repository:** [github.com/ulissesflores/lgbt-indicador-violencia](https://github.com/ulissesflores/lgbt-indicador-violencia) ·
**License:** this notebook and the code are Apache-2.0; data and text are CC BY 4.0 (see `NOTICE`).

This notebook checks the replication package of the article from the public repository, with no local setup:

- **(a)** clone the repository at the version the article cites (`REF` below);
- **(b)** install what the package needs (`requirements.txt` and `pytest`);
- **(c)** verify the SHA-256 seal of the code, documents and derived data (`make_provenance.py --verify`);
- **(d)** run `run_all.py`: integrity, recomputation of every derived output in a scratch copy (`.run/replica`)
  compared with the committed files, the tests that lock the published numbers, and the seal again (Track 1 of
  `REPRODUCIBILITY.md`);
- **(e)** check every number the article prints against the repository file that holds it
  (`verify_paper_numbers.py`, `tests/test_paper_numbers.py`) and show them side by side: article value, value in the
  committed file, match, and, when step (d) rewrote that file, the value it printed;
- **(f)** optional: download the 2019 National Health Survey microdata from IBGE (about 30 MB, 455 MB unzipped), check
  its SHA-256 and rerun the five scripts that read it (part of Track 2).

Steps (a) to (e) take about 5 minutes and need no data outside the repository. Step (f) takes about 2 minutes more.

**What this notebook cannot reproduce, and why** (full list: `REPRODUCIBILITY.md`, "What cannot be re-run")

- **The rest of Track 2.** Mortality (SIM/DATASUS, about 1.6 GB), notifications (Sinan, about 300 MB), electoral
  records (TSE) and the Disque 100 hotline (about 13 GB) are downloaded by `scripts/acquire_*.py` on demand; the DATASUS
  `.dbc` files also need `pyreaddbc` and `dbfread`. Step (d) recomputes everything derived from their committed
  outputs.
- **Material that is not published**: the nominal Grupo Gay da Bahia list and the news texts behind it (victims'
  privacy), restricted or copyrighted PDFs read for the literature, and full parliamentary speeches.
- **Readings by language models** (third coder, second symmetry reading): they need an authenticated command line and
  reply differently each time; the recorded coding is in the repository.
- **Searches and downloads made on given days** (news, media counts, budget, prices, governments' statements): the
  sources change; the repository holds the counts, the dated SHA-256 of each raw file and the parsing scripts.
- **Numbers with no file to check.** Design parameters, declared limits, conditional statements and sources not yet
  opened are listed in `paper_numbers.json` with a stated reason and counted at the end of step (e); they are not
  counted as matches.

## Settings

`REF` is the version checked. Set it to the tag the article cites so that this notebook checks exactly that version;
`"main"` checks the latest version on GitHub. `RUN_PNS = True` turns on step (f).

In [ ]:
# SPDX-License-Identifier: Apache-2.0
# Copyright 2026 Carlos Ulisses Flores (Codex Hash Ltda.)
# Settings and the one helper every later cell uses to run a command.
import html
import os
import subprocess
import sys
from pathlib import Path

REF = "main"  # the article's tag, e.g. "v1.3.0"; "main" = latest
REPO_URL = "https://github.com/ulissesflores/lgbt-indicador-violencia.git"
RUN_PNS = False  # step (f): True downloads the PNS 2019 microdata and reruns its five scripts

# Overrides for automated test runs of this notebook only; a reader leaves them unset.
REPO_URL = os.environ.get("LGBT_NB_REPO", REPO_URL)
REF = os.environ.get("LGBT_NB_REF", REF)
RUN_PNS = RUN_PNS or os.environ.get("LGBT_NB_RUN_PNS") == "1"
DEFAULT_WORK = "/content" if Path("/content").is_dir() else Path.cwd()
WORK = Path(os.environ.get("LGBT_NB_WORKDIR", DEFAULT_WORK))
PKG = WORK / "lgbt-indicador-violencia"

# Nothing may write bytecode into the sealed folders; the notebook's Python runs every script.
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]


def sh(cmd, cwd=None):
    """Run a command, print its output as it comes, and stop the notebook if it fails."""
    print("$", " ".join(str(c) for c in cmd), flush=True)
    lines = []
    with subprocess.Popen(
        cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    ) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            lines.append(line)
    if proc.returncode:
        raise RuntimeError(f"exit code {proc.returncode}: {' '.join(str(c) for c in cmd)}")
    return "".join(lines)


print(f"REF={REF}  WORK={WORK}  RUN_PNS={RUN_PNS}  Python {sys.version.split()[0]}")

## (a) Clone the repository at `REF`

In [ ]:
# A fresh clone every time, checked out at REF; the working tree must be clean before anything runs.
import shutil

if PKG.exists():
    shutil.rmtree(PKG)
sh(["git", "clone", "--quiet", REPO_URL, str(PKG)])
sh(["git", "-c", "advice.detachedHead=false", "checkout", "--quiet", REF], cwd=PKG)
COMMIT = sh(["git", "rev-parse", "HEAD"], cwd=PKG).strip()
assert sh(["git", "status", "--porcelain"], cwd=PKG) == "", "the checkout is not clean"
print(f"checked out {REF} = {COMMIT}")

## (b) Install what the package needs

In [ ]:
# numpy, pandas, scipy and matplotlib (Track 1), plus pytest for step (e).
sh([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt", "pytest"], cwd=PKG)
sh([sys.executable, "-m", "pytest", "--version"], cwd=PKG)

## (c) Verify the seal

One SHA-256 `chain_hash` covers the code, the dated documents, the derived data and the tests
(`output/hash-chain.md` lists every file). This recomputes it from the files and compares it with the committed one.

In [ ]:
# Recompute the chain hash from the files and compare it with output/provenance.json.
sh([sys.executable, "-I", "make_provenance.py", "--verify"], cwd=PKG)

## (d) Recompute, test, verify the seal again

`run_all.py` copies the repository to `.run/replica`, reruns every Track 1 script there, compares each file a script
rewrote with the committed one (byte for byte, or numerically equal for the few outputs whose last digits depend on
the platform; images are informative), runs the tests that lock the published numbers and verifies the seal.
About 5 minutes.

In [ ]:
# Track 1 end to end; the last lines summarise the recomputation and the tests.
RUN_ALL = sh([sys.executable, "-I", "run_all.py"], cwd=PKG)

## (e) Check every number the article prints

`paper_numbers.json` maps each number printed in the article (Brazilian Portuguese edition, the one the other editions
are translated from) to the file and place that hold it, with the SHA-256 of the file and of the script that writes it.
The verifier checks those hashes and every value; the test does the same inside `pytest`.

In [ ]:
# The package's own checks first: they stop the notebook if any printed number fails.
sh(
    [sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_paper_numbers.py"],
    cwd=PKG,
)
VERIFY = sh([sys.executable, "-I", "verify_paper_numbers.py"], cwd=PKG)

In [ ]:
# Side by side, number by number, using the verifier's own functions (imported unchanged).
import csv

from IPython.display import HTML, display

sys.path.insert(0, str(PKG))
import verify_paper_numbers as V  # noqa: E402

REPLICA = PKG / ".run" / "replica"


def rewritten(rel):
    """Tell whether step (d) rewrote this file in the replica (the copy keeps the timestamps)."""
    rep = REPLICA / rel
    return rep.is_file() and rep.stat().st_mtime_ns != (PKG / rel).stat().st_mtime_ns


def judge(value, p, scale):
    """Apply the verifier's rule: a corrected number must round to the correction."""
    if p.get("corrected"):
        return V.bate(value, p["corrected"], scale)
    return V.bate(value, p["value"], scale, p.get("bound"))


def rows():
    """One row per printed number with a place in the repository; counts for the rest."""
    table, out, unlocated = V.carrega(), [], {}
    for e in table["numbers"]:
        if e["class"] not in V.CLASSES_COM_LOCAL:
            unlocated[e["class"]] = unlocated.get(e["class"], 0) + len(e["printed"])
            continue
        rel, scale = e["locator"]["file"], e.get("scale", 1.0)
        value = V.resolve(e["locator"], PKG)
        again = V.resolve(e["locator"], REPLICA) if rewritten(rel) else None
        for p in e["printed"]:
            fix = p.get("corrected")
            printed = p["value"] + (f" -> {fix} (correction)" if fix else "")
            rerun = "not rewritten (committed file, checked by hash)"
            if again is not None:
                rerun = f"{again}: {'match' if judge(again, p, scale) else 'NO MATCH'}"
            out.append(
                (
                    p["line"],
                    p["where"],
                    e["class"],
                    printed,
                    str(value),
                    judge(value, p, scale),
                    rel,
                    rerun,
                )
            )
    return out, unlocated


ROWS, UNLOCATED = rows()
ok = sum(r[5] for r in ROWS)
regen = [r for r in ROWS if not r[7].startswith("not rewritten")]
regen_bad = [r for r in regen if r[7].endswith("NO MATCH")]
print(f"{ok} of {len(ROWS)} printed numbers with a place in the repository match their file.")
print(f"{len(regen)} come from a file step (d) rewrote; {len(regen) - len(regen_bad)} match again.")
for cls, n in sorted(UNLOCATED.items()):
    print(f"  {n:>5} printed numbers of class {cls!r}: listed with a reason, not checked")

head = (
    "Article line",
    "Where",
    "Class",
    "Article value",
    "Committed file",
    "Match",
    "File",
    "Rewritten in step (d)",
)
cells = "".join(
    "<tr>"
    + "".join(
        f"<td>{'yes' if c is True else 'NO' if c is False else html.escape(str(c))}</td>" for c in r
    )
    + "</tr>"
    for r in ROWS
)
display(
    HTML(
        "<div style='max-height:420px;overflow:auto'><table><tr>"
        + "".join(f"<th>{h}</th>" for h in head)
        + f"</tr>{cells}</table></div>"
    )
)
with open(WORK / "paper_numbers_check.csv", "w", newline="", encoding="utf-8") as fh:
    csv.writer(fh).writerows([head, *ROWS])
print(f"full table saved to {WORK / 'paper_numbers_check.csv'}")
assert ok == len(ROWS), "some printed numbers do not match the committed file"
assert not regen_bad, "a recomputed file holds a value that does not match the article"

## (f) Optional: the 2019 National Health Survey microdata

Set `RUN_PNS = True` in the settings cell and run this cell. `scripts/acquire_pns2019.py` downloads the microdata and
layout from IBGE, checks the SHA-256 recorded in `manifest/MANIFESTO-FONTES.csv` (a mismatch means a different release
from the one used) and unzips them into `sources/public/` (ignored by git). The five scripts that read them then rewrite
their outputs in place, and each output is compared with the committed one.

One expected difference: `data/ggb/c08_cenarios_marginais.csv` has a column `ancoras_conferidas` that records whether
the Grupo Gay da Bahia report texts (third-party, not distributed) were on the machine when the script ran. In a clone
it turns from `True` to `False`; every number in the file stays the same, and the comparison below ignores only that
column.

In [ ]:
# Download, check and rerun the five PNS scripts; compare every output with the committed file.
import csv as _csv
import filecmp

PNS_SCRIPTS = [
    "data/pns/seguro_pns2019.py",
    "data/pns/agressor_risco_pns2019.py",
    "data/pns/pns_c05_razao_direta.py",
    "data/ggb/cobertura_implicita.py",
    "data/ggb/ggb_c08_numeradores.py",
]
PNS_OUTPUTS = [
    f"{base}.{ext}"
    for base in (
        "data/pns/seguro_pns2019",
        "data/pns/agressor_risco_pns2019",
        "data/pns/c05_razao_direta",
        "data/ggb/cobertura_implicita",
        "data/ggb/c08_numeradores",
    )
    for ext in ("md", "csv")
] + ["data/ggb/c08_cenarios_marginais.csv"]
IGNORED_COLUMN = {"data/ggb/c08_cenarios_marginais.csv": "ancoras_conferidas"}


def same_without(a, b, column):
    """Compare two CSV files cell by cell, ignoring one named column."""
    ra, rb = (list(_csv.DictReader(open(p, encoding="utf-8"))) for p in (a, b))
    strip = lambda rs: [{k: v for k, v in r.items() if k != column} for r in rs]  # noqa: E731
    return strip(ra) == strip(rb)


if RUN_PNS:
    BEFORE = WORK / "pns_committed"
    shutil.rmtree(BEFORE, ignore_errors=True)
    for rel in PNS_OUTPUTS:
        (BEFORE / rel).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(PKG / rel, BEFORE / rel)
    sh([sys.executable, "-I", "scripts/acquire_pns2019.py"], cwd=PKG)
    for script in PNS_SCRIPTS:
        sh([sys.executable, "-I", script], cwd=PKG)
    bad = []
    for rel in PNS_OUTPUTS:
        if filecmp.cmp(BEFORE / rel, PKG / rel, shallow=False):
            print(f"identical: {rel}")
        elif rel in IGNORED_COLUMN and same_without(BEFORE / rel, PKG / rel, IGNORED_COLUMN[rel]):
            print(f"identical except column {IGNORED_COLUMN[rel]!r} (expected): {rel}")
        else:
            print(f"DIFFERENT: {rel}")
            bad.append(rel)
    assert not bad, f"rerun printed something other than the committed file: {bad}"
else:
    print("RUN_PNS is False: step (f) skipped")